## Setup

In [ ]:
# Allows importing from local files
import sys

# Add the relative path to project root
sys.path.append('../../../')
%load_ext autoreload
%autoreload 2 

In [ ]:
from model_tuning_workspace.prompting.system_msg import sys_msg

PATH_TO_DATASET = ".jsonl" # TODO: Add path to JSONL file
MODEL_TO_FT = "meta-llama/Llama-3.1-8B-Instruct"

MODEL_NAME = "NVDRS"
VERSION = 0.2


In [ ]:
import os
import getpass

hf_token = os.environ['HF_TOKEN']
if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face API token: ")

In [ ]:
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## Load the model

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 5500

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_TO_FT,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None, 
    load_in_4bit=False,
    token=hf_token,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)
print(model.print_trainable_parameters())

## Prepare the dataset

In [ ]:
from model_tuning_workspace.training.data import convert_jsonl_to_dataset

d_set, dataset_text_field = convert_jsonl_to_dataset(PATH_TO_DATASET, MODEL_TO_FT)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=d_set,
    dataset_text_field=dataset_text_field,
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_steps=3,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=4567,
        output_dir="outputs",
    ),
)

## Training

In [ ]:
trainer_stats = trainer.train()

## Try inference

In [ ]:
from transformers import TextStreamer


model = FastLanguageModel.for_inference(model)
messages = [
    {
        "role": "system",
        "content": sys_msg,
    },
    {
        "role": "user",
        "content": "hello",
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=700, use_cache=True)

## Push the model to the hub

In [ ]:
commit_msg = "Initial commit"
commit_desc = "Initial commit of the"

final_model_name = f"{MODEL_NAME}-v{VERSION}"

# Saves just the LoRA adapter
model.push_to_hub(
    f"{final_model_name}_lora",
    commit_message=commit_msg,
    commit_description=f"{commit_desc} adapter",
    token=hf_token,
    private=True,
)

# Saves 16 bit merged model
model.push_to_hub_merged(
    final_model_name,
    tokenizer,
    save_method="merged_16bit",
    commit_message=commit_msg,
    commit_description=f"{commit_desc} model",
    token=hf_token,
    private=True,
)